# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I use the starter dataset at one row per content item. The target is `is_declining_label`, defined as `trend_direction == "down"`.

The model features are the starter numeric and categorical features. Numeric traffic totals are log-transformed because they are heavy-tailed. Missing numeric values are filled with 0 and missing categorical values with `"unknown"`.

I do not use `content_id` or `client_id` as features. They are identifiers used only for grouping and validation.


In [7]:
!git clone https://github.com/Alpeshmore/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 138 (delta 50), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.85 MiB | 4.57 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [11]:
import pandas as pd
import numpy as np

DATA_PATH = "flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [12]:


# Same filtering used by the starter preparation step
df = df[
    (df["impressions_90d"] > 0) &
    (df["content_age_days"] >= 90)
].copy()

df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

# Engineered numeric features
for col in [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ai_sessions_90d",
]:
    df[f"log_{col}"] = np.log1p(
        pd.to_numeric(df[col], errors="coerce").fillna(0)
    )

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "log_impressions_90d",
    "log_clicks_90d",
    "log_sessions_90d",
    "log_ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier",
]

# Fill numeric missing values
for col in numeric_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").replace(
        [np.inf, -np.inf], np.nan
    ).fillna(0)

# Fill categorical missing values
for col in categorical_features:
    df[col] = (
        df[col]
        .fillna("unknown")
        .astype(str)
        .replace({"": "unknown", "nan": "unknown"})
    )

# Target
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

feature_columns = numeric_features + categorical_features

X = df[feature_columns].copy()
y = df["is_declining_label"].copy()

print("Rows:", len(df))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(feature_columns))
print("Positive label rate:", round(y.mean(), 3))
print("Feature matrix shape:", X.shape)


Rows: 30000
Numeric features: 18
Categorical features: 8
Total model features: 26
Positive label rate: 0.542
Feature matrix shape: (30000, 26)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The numeric features describe search demand, content size, observed traffic, engagement, position, age, freshness, and AI-referral activity. The log features are transformations of 90-day totals, not new information.

Categorical features describe content and performance tiers.

For this starter task, these values are calculated from the available 90-day/current-window snapshot. They are available at the prediction point used by the starter model, but this is important: the starter label is also calculated from the same current window. Therefore, this is a proxy exercise rather than a true future-outcome prediction.

Numeric missing values are filled with 0 and categorical missing values with `"unknown"`. This is the starter pipeline's treatment. The data dictionary warns that missingness is systematic, especially for keyword fields, so the zero fill should not be interpreted as meaning that the underlying measurement was truly zero.

`content_id` and `client_id` are not model features. They are identifiers for grouping and validation only.


In [13]:
feature_notes = pd.DataFrame({
    "feature": feature_columns,
    "type": (
        ["numeric"] * len(numeric_features)
        + ["categorical"] * len(categorical_features)
    ),
    "missing_handling": (
        ["numeric -> 0"] * len(numeric_features)
        + ["categorical -> 'unknown'"] * len(categorical_features)
    ),
    "available_before_prediction": ["yes"] * len(feature_columns),
})

display(feature_notes)

print("\nMissing values remaining in X:")
print(X.isna().sum().sum())

print("\nFeature availability check:")
print("All selected features present:", set(feature_columns).issubset(df.columns))

,feature,type,missing_handling,available_before_prediction
0,search_volume,numeric,numeric -> 0,yes
1,competition,numeric,numeric -> 0,yes
2,cpc,numeric,numeric -> 0,yes
3,word_count,numeric,numeric -> 0,yes
4,char_count,numeric,numeric -> 0,yes
5,log_impressions_90d,numeric,numeric -> 0,yes
6,log_clicks_90d,numeric,numeric -> 0,yes
7,log_sessions_90d,numeric,numeric -> 0,yes
8,log_ai_sessions_90d,numeric,numeric -> 0,yes
9,days_with_impressions,numeric,numeric -> 0,yes



Missing values remaining in X:
0

Feature availability check:
All selected features present: True


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature vector for three leakage risks.

First, `trend_direction` is directly used to create the label, so it is excluded. `trend_pct` is also excluded because it is used to define `trend_direction`.

Second, the starter dataset uses a trailing 90-day snapshot and derives the label from the current 30-day comparison. Therefore the starter label is not a clean future-window outcome. I treat this model as a proxy exercise, not as proof of future prediction skill.

Third, I checked for product decision fields such as `health_score`, `priority_score`, `action_type`, and refresh flags. These are not used as model features. If they were available, they would represent decisions made by an existing system rather than independent observed signals.

The main leakage risk is therefore the overlap between the current feature window and the current-window label. A future-looking version should use a feature window that ends before the future target window begins.


In [14]:
# 1. Direct and sibling label leakage
label_columns = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
]

leaky_label_columns = [
    col for col in label_columns
    if col in feature_columns
]

print("Label-derived columns found in features:", leaky_label_columns)
assert not leaky_label_columns, "Label-derived leakage detected!"

# 2. IDs must not be model features
id_columns = ["content_id", "client_id"]

id_leaks = [
    col for col in id_columns
    if col in feature_columns
]

print("ID columns used as features:", id_leaks)
assert not id_leaks, "Identifier leakage detected!"

# 3. Product decision fields should not be features
product_decision_columns = [
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "needs_ctr_fix",
    "is_quick_win",
]

product_leaks = [
    col for col in product_decision_columns
    if col in feature_columns
]

print("Product decision fields used as features:", product_leaks)
assert not product_leaks, "Product-decision leakage detected!"

# 4. Explicit final audit
print("\nLeakage audit passed:")
print("- No label-derived columns:", not leaky_label_columns)
print("- No IDs as features:", not id_leaks)
print("- No product decision fields:", not product_leaks)

# Important timeline note
print("\nTimeline note:")
print("Starter label = current-window trend.")
print("Therefore this is a proxy/current-window classification task,")
print("not a strict prior-window -> future-window prediction.")

Label-derived columns found in features: []
ID columns used as features: []
Product decision fields used as features: []

Leakage audit passed:
- No label-derived columns: True
- No IDs as features: True
- No product decision fields: True

Timeline note:
Starter label = current-window trend.
Therefore this is a proxy/current-window classification task,
not a strict prior-window -> future-window prediction.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* `content_id` — pseudonymous identifier; useful for grouping, not a meaningful predictive signal.
* `client_id` — identifies the client group; using it as a feature could let the model memorize client-specific patterns.
* `trend_direction` — directly defines the target, so using it would reveal the answer.
* `trend_pct` — used to derive `trend_direction`, so it is label-derived.
* `is_declining_label` — this is the target itself and cannot be an input feature.
* `provider_used` — content-generation metadata is not part of the starter model feature vector.
* `model_used` — content-generation metadata is not part of the starter model feature vector.
* `health_score` — product decision output; using it would reproduce an existing decision.
* `priority_score` — product decision output; not an independent observed signal.
* `action_type` — existing product decision; would create a circular result.
* `refresh_tier` / refresh flags — existing decision logic rather than independent evidence.


In [15]:
excluded_fields = {
    "content_id": "Identifier; grouping only.",
    "client_id": "Client grouping; not a predictive feature.",
    "trend_direction": "Direct source of the target.",
    "trend_pct": "Used to derive the target direction.",
    "is_declining_label": "Target itself.",
    "provider_used": "Not part of the starter model feature vector.",
    "model_used": "Not part of the starter model feature vector.",
    "health_score": "Existing product decision; circular if used as a feature.",
    "priority_score": "Existing product decision; circular if used as a feature.",
    "action_type": "Existing product decision; not independent evidence.",
    "refresh_tier": "Existing decision logic; excluded from modeling.",
}

excluded_check = pd.DataFrame(
    [
        {
            "field": field,
            "reason": reason,
            "present_in_data": field in df.columns,
            "used_as_feature": field in feature_columns,
        }
        for field, reason in excluded_fields.items()
    ]
)

display(excluded_check)

assert not any(
    excluded_check["used_as_feature"]
), "An excluded field has entered the feature vector."

print("Excluded-field audit passed.")

,field,reason,present_in_data,used_as_feature
0,content_id,Identifier; grouping only.,True,False
1,client_id,Client grouping; not a predictive feature.,True,False
2,trend_direction,Direct source of the target.,True,False
3,trend_pct,Used to derive the target direction.,True,False
4,is_declining_label,Target itself.,True,False
5,provider_used,Not part of the starter model feature vector.,True,False
6,model_used,Not part of the starter model feature vector.,True,False
7,health_score,Existing product decision; circular if used as...,False,False
8,priority_score,Existing product decision; circular if used as...,False,False
9,action_type,Existing product decision; not independent evi...,False,False


Excluded-field audit passed.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.